# Conformal interval implementation

This page describes how the conformal methods are built in code. The statistical rules are on the [](../methodology/conformal.ipynb) page. In brief, split conformal takes the k-th smallest absolute calibration residual as the half-width of a band around the prediction, so its width is constant; CV+ (cross-validation plus) takes its endpoints from out-of-fold predictions shifted by out-of-fold residuals, so its width need not be constant; CQR (conformalized quantile regression) fits two quantile models and widens or narrows their band by a conformal correction; the distributional route takes its interval from the predictive distribution of an LSS model.

## Module layout

`python/mboostrs/conformal.py` holds the orchestration for every method: `conformal_interval` dispatches to `_split_conformal`, `_cv_plus`, `_cqr` and `_distributional`, and each one refits models on subsets of the training rows. The numerical helpers are in Rust (`crates/mboostrs-core/src/conformal.rs`) and reach Python through `mboostrs._rust`. The Rust helpers cover the conformal quantile, the CV+ bounds and ranks, and the tuned step-count rules (`tuned_mstop_start`, `tuned_mstop_next`, `tuned_mstop_choice`). Python does not implement any of these numerical rules.

## Calibration responses

`_response(model, data)` returns the observed response in original units. It reads the column named by `_original_response_name(model)`. For a transformed model, that is the source column of the transform, such as `y` for `center(y)`, not the transformed name. A calibration call that does not supply that column raises `ValueError` with the column name. Every calibration score is built from `_response` output and from predictions mapped back to original units by `_predict_raw`.

## Tuned quantile refit

`_quantile_refit(model, rows, tau, seed, mstop)` builds the quantile model at level `tau` from the `Quantile` family. A fixed integer `mstop` refits at that step count. The value `"tune"` calls `_cv_tuned_refit`, which runs the search described in the methodology chapter:

- `_cv_tuned_refit` takes the start from `conformal_tuned_mstop_start`, cross-validates with `cv_weights(..., "kfold", _TUNE_FOLDS, seed)`, and doubles through `conformal_tuned_mstop_next`, up to `_QUANTILE_MSTOP_CAP` (10,000). It chooses the step count with `conformal_tuned_mstop_choice`, which never returns zero.
- The cap warning goes through `_TUNING_CAPS`, a context variable that `_tuning_warnings()` sets for each public call. It makes the warning appear once per call, even when both quantile models reach the cap.
- The search keeps one fit on the proper-training rows and continues it to each new grid end with `continue_to`. A continued fit gives exactly the same path as a fresh fit to the same length, so the search does not refit at each doubling.
- The cross-validation call uses the mode the response needs. For a model without a response transform it is `"refit"`, which relearns each fold's preprocessing from its own training rows. For a GAM, `cvrisk` receives `_path_cache`, a dict that keeps each fold's fitted model and its path predictions between grid ends. The next call continues each stored fold fit and predicts only the steps it has not yet seen, which is exact because a boosting path is prefix-consistent (the first k steps of a longer path equal the fit with k steps). GLM models have no such cache and are cross-validated from their fit at each end.
- For a model with a response transform the mode is `"fixed"`, which uses the constants learned on the proper-training rows for every fold. The refit mode refuses `_extend` (the call that resumes a fitted path to a longer step count), so the search cannot resume the folds through that path.

## Choosing the preprocessing mode

`gamboost.cvrisk(folds, preprocessing=...)` takes two modes. `"fixed"` reuses the full-data transformations and spline knots, as mboost does. `"refit"` learns the transformations, factor encodings and spline boundaries on each fold's training rows and applies them to that fold's held-out rows.

A fold-fixed transform is a single set of constants, so its risks share one scale. The per-fold refit gives each fold its own constants, so its risks can be compared only if the transform is the same in every fold. For a centred or scaled response, `_cv_tuned_refit` uses `"fixed"` for this reason. For an untransformed response the per-fold recipe has no constants to differ, so `"refit"` is kept, and the untransformed path is unchanged.

## Model types

`_refit` builds the quantile fit from the model's own specification and the chosen rows, for both `GAMBoostModel` and `GLMBoostModel`. The distributional route for `LSSModel` is `_distributional`, a separate path that does not use the CQR refit in the section Tuned quantile refit.

## Tests

These files cover the behaviour in this chapter and run without the R package:

- `tests/test_conformal_response_units_652.py`: original-unit calibration for `center`, `scale`, `log` and other transforms, for split, CV+ and CQR; external calibration rows; and tuned runs on a fold-fixed transform for every method.
- `tests/test_conformal_core.py`: the Rust helpers against a Python oracle, including `conformal_tuned_mstop_*`.
- `tests/test_conformal_cqr_warnings.py`: the cap warning fires once and the search range warning does not leak.
- `tests/test_conformal_mstop_tuning.py` and `tests/test_conformal_guarantee_contract.py`: tuned fits do not depend on calibration responses, and the guarantee wording matches the implementation.

The benchmark behind the table Measured coverage and width with transformed responses in [](../methodology/conformal.ipynb) is `docs/book/benchmarks/interval_calibration/transform_tuning.py`, which writes `transform_tuning.json`.